In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

#1. Generate 200 random customers
np.random.seed(42)
spend = np.random.uniform (20,150,200) # Spend between $20 and $150
calls = np.random.randint (0,10,200) # 0 to 9 support calls
months = np.random.randint (1,60,200) # 1 to 59 months subscribed

# Secret Logic: High calls and high spend = Likely to churn. Long subscription =
churn_probability = (calls * 0.15) + (spend * 0.005) - (months * 0.02)
# Convert the math into strict 1s and 0s
churned = np.where(churn_probability > np.median(churn_probability), 1, 0)

# Put it in a Pandas DataFrame
df_churn = pd.DataFrame({
    'Monthly_Spend': spend,
    'Support_Calls': calls,
    'Months_Subscribed': months,
    'Churned': churned
})
df_churn

,Monthly_Spend,Support_Calls,Months_Subscribed,Churned
0,68.690215,7,33,1
1,143.592860,3,33,1
2,115.159212,0,51,0
3,97.825603,7,43,1
4,40.282423,3,37,0
...,...,...,...,...
195,65.397245,0,6,0
196,114.374238,3,58,0
197,136.624334,7,44,1
198,135.321235,1,45,0


In [2]:
#2. Prepare X (Features) and y (Target)
X = df_churn[['Monthly_Spend', 'Support_Calls', 'Months_Subscribed']]
y = df_churn['Churned']

#3. Train/Test Split (80% Train, 20% Test)
X_train,X_test,y_train,y_test = train_test_split (X, y, test_size = 0.2, random_state = 42)

#4. Train the Classification Model
model_clf = LogisticRegression()
model_clf.fit(X_train,y_train)

#5. Predict on the hidden test data
predictions = model_clf.predict(X_test)

#6. Grade the model
accuracy = accuracy_score(y_test, predictions)
print("--- MODEL REPORT CARD ---")
print(f"Classification Accuracy: {accuracy * 100:0.1f}%\n")

#let's see the first 5 predictions vs the actual reality!
results_df = pd.DataFrame({
    'Model Predicted': predictions[:5],
    'Actual Reality': y_test[:5].values
})
print("First 5 Test customers:")
print(results_df)

--- MODEL REPORT CARD ---
Classification Accuracy: 95.0%

First 5 Test customers:
   Model Predicted  Actual Reality
0                0               0
1                1               1
2                0               0
3                1               1
4                1               1


In [3]:
from sklearn.metrics import confusion_matrix

# 1. Generate the raw confusion matrix numbers
cm = confusion_matrix(y_test, predictions)

# 2. Put it into a Pandas DataFrame so we can add labels to the rows and columns
cm_df = pd.DataFrame(
    cm, 
    columns=['Model Predicted: STAY (0)', 'Model Predicted: CHURN (1)'], 
    index=['Actual Reality: STAY (0)', 'Actual Reality: CHURN (1)']
)

print("--- CONFUSION MATRIX ---")
print(cm_df)

--- CONFUSION MATRIX ---
                           Model Predicted: STAY (0)  \
Actual Reality: STAY (0)                          17   
Actual Reality: CHURN (1)                          1   

                           Model Predicted: CHURN (1)  
Actual Reality: STAY (0)                            1  
Actual Reality: CHURN (1)                          21  


In [4]:
# 1. Create a DataFrame for our new customer (columns MUST match your training data)
new_customer = pd.DataFrame({
    'Monthly_Spend': [85.50],
    'Support_Calls': [6],
    'Months_Subscribed': [3]
})

# 2. Make the strict prediction (0 = Stay, 1 = Churn)
prediction = model_clf.predict(new_customer)

# 3. THE SUPERPOWER: Predict the exact probability!
probability = model_clf.predict_proba(new_customer)

print("--- NEW CUSTOMER REPORT ---")
if prediction[0] == 1:
    print("Action Required: ALERT! This customer is likely to CHURN (Cancel).")
else:
    print("Action Required: Relax. This customer is likely to STAY.")

# predict_proba returns two numbers: [Chance of 0, Chance of 1]
# We want the second number (index 1), which is the chance of Churn.
churn_risk = probability[0][1] * 100
print(f"Exact Churn Risk: {churn_risk:.1f}%")

--- NEW CUSTOMER REPORT ---
Action Required: ALERT! This customer is likely to CHURN (Cancel).
Exact Churn Risk: 100.0%
